# Ajuster m2m100 au pidgin camerounais de KAMA (Colab)

Base : `masakhane/m2m100_418M_en_pcm_rel_news_ft` (licence **AFL-3.0**). Les donn?es MT560 sont en **CC BY 4.0**.

**Avant de lancer :**
1. Mets **`KAMA_pidgin_m2m_drive.zip`** dans Mon Drive (racine ou sous `KAMA/`), sans le d?compresser. La cellule 2 le trouve toute seule.
2. **Ex?cution ? Modifier le type d'ex?cution ? GPU T4**, avec **Python 3.12** (pas ? Latest ?).
3. Lance d'abord avec `SMOKE = True` : 300 paires, 30 pas, puis 5 traductions de contr?le dont `ZZ1ZZ`. Note le **temps par pas**.
4. Si tout va bien, mets `SMOKE = False` et relance la cellule 1, puis les sections 4 ? 6. Le vrai entra?nement fait **3 passes**.

Point critique : la sortie pidgin de ce mod?le passe par le jeton `__fr__` (`forced_bos_token_id=128028`) et la source par `__en__`. Ne remplace pas `__fr__` par un code de langue pidgin : le mod?le perdrait ce qu'il sait d?j?.

## 1. R?glages

`SMOKE = True` ne sert qu'? v?rifier que la cha?ne tourne. Le vrai entra?nement utilise toutes les paires, 3 passes, et garde le meilleur point selon chrF sur `dev`.

In [ ]:
import sys
assert sys.version_info[:2] <= (3, 12), f"Python {sys.version.split()[0]} : choisis Python 3.12 dans le type d'ex?cution"
print("Python", sys.version.split()[0], "OK")

SMOKE = True          # True : essai rapide. False : vrai entra?nement.
EPOCHS = 3            # passes du vrai entra?nement
MODELE = "masakhane/m2m100_418M_en_pcm_rel_news_ft"
FORCED_BOS = 128028   # __fr__ : langue de sortie du pidgin pour ce mod?le

MON_DRIVE = "/content/drive/MyDrive"
LOCAL = "/content/KAMA/pidgin_m2m"
SORTIE = "/content/m2m100_sortie" + ("_smoke" if SMOKE else "")
SORTIE_DRIVE = MON_DRIVE + "/KAMA/pidgin_m2m_sorties"
print("mode", "ESSAI (SMOKE)" if SMOKE else f"ENTRAINEMENT COMPLET, {EPOCHS} passes")

## 2. Drive et donn?es

Le zip est cherch? dans la session Colab, puis dans Mon Drive, puis propos? en envoi manuel. Il est d?compress? sur le disque de Colab (plus rapide que Drive).

In [ ]:
import os, glob, shutil, zipfile
os.chdir("/content")

zips = glob.glob("/content/KAMA_pidgin_m2m_drive.zip")
if not zips:
    try:
        if not os.path.isdir(MON_DRIVE):
            from google.colab import drive
            drive.mount("/content/drive", timeout_ms=600000)
        zips = glob.glob(MON_DRIVE + "/KAMA_pidgin_m2m_drive.zip") + glob.glob(MON_DRIVE + "/KAMA/**/KAMA_pidgin_m2m_drive.zip", recursive=True)
    except Exception as e:
        print("Drive non mont? :", type(e).__name__, "-> envoi manuel du zip")

if not zips:
    from google.colab import files
    up = files.upload()
    assert "KAMA_pidgin_m2m_drive.zip" in up, "KAMA_pidgin_m2m_drive.zip attendu"
    zips = ["/content/KAMA_pidgin_m2m_drive.zip"]

print("zip utilis? :", zips[0])
shutil.rmtree("/content/KAMA", ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    assert z.testzip() is None, "zip ab?m? : renvoie-le depuis ton ordinateur"
    z.extractall("/content")

for f in ("data/train.jsonl", "data/dev.jsonl", "data/test_soreva.jsonl"):
    assert os.path.exists(f"{LOCAL}/{f}"), f"manquant dans le zip : {f}"

import json, collections
train = [json.loads(l) for l in open(f"{LOCAL}/data/train.jsonl", encoding="utf-8") if l.strip()]
dev = [json.loads(l) for l in open(f"{LOCAL}/data/dev.jsonl", encoding="utf-8") if l.strip()]
test = [json.loads(l) for l in open(f"{LOCAL}/data/test_soreva.jsonl", encoding="utf-8") if l.strip()]
print("train", len(train), "dev", len(dev), "test SOREVA", len(test))
print("sources train", collections.Counter(x["source"] for x in train))
print("sources dev", collections.Counter(x["source"] for x in dev))

## 3. Installation

Versions align?es sur les autres carnets KAMA. Si pip signale un conflit, red?marre l'ex?cution puis relance les cellules 1 et 2.

In [ ]:
!pip install -q "transformers==4.57.6" sentencepiece sacremoses sacrebleu accelerate ctranslate2
!nvidia-smi -L

## 4. Entra?nement

`load_best_model_at_end` garde le meilleur point selon **chrF** sur `dev`. En mode `SMOKE`, le temps par pas est affich? apr?s les 30 pas ; multiplie-le par le nombre de pas d'une passe pour estimer le vrai entra?nement.

In [ ]:
import json, math, random, time
from pathlib import Path

import numpy as np
import sacrebleu
import torch
from torch.utils.data import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

random.seed(13)
torch.manual_seed(13)
DATA = Path(LOCAL) / "data"
SORTIE = Path(SORTIE)
SORTIE.mkdir(parents=True, exist_ok=True)

tok = M2M100Tokenizer.from_pretrained(MODELE)
tok.src_lang = "en"
tok.tgt_lang = "fr"          # les labels commencent par __fr__
modele = M2M100ForConditionalGeneration.from_pretrained(MODELE)
modele.generation_config.forced_bos_token_id = FORCED_BOS
modele.generation_config.num_beams = 4
modele.generation_config.no_repeat_ngram_size = 3
modele.generation_config.max_length = 128
print("mod?le", MODELE, "| param?tres", sum(p.numel() for p in modele.parameters()) / 1e6, "M")

def lire(nom, limite=None):
    lignes = [json.loads(l) for l in open(DATA / nom, encoding="utf-8") if l.strip()]
    return lignes[:limite] if limite else lignes

class Paires(Dataset):
    def __init__(self, lignes):
        self.exemples = [tok(l["en"], text_target=l["wes"], truncation=True, max_length=128) for l in lignes]
    def __len__(self):
        return len(self.exemples)
    def __getitem__(self, i):
        return self.exemples[i]

train_lignes = lire("train.jsonl", 300 if SMOKE else None)
dev_lignes = lire("dev.jsonl", 60 if SMOKE else None)
test_lignes = lire("test_soreva.jsonl")
train, dev, test = Paires(train_lignes), Paires(dev_lignes), Paires(test_lignes)
print("paires utilis?es :", len(train), "train,", len(dev), "dev,", len(test), "test")

chrf = sacrebleu.CHRF(word_order=2)
def mesures(p):
    preds, labels = p
    preds = np.where(preds != -100, preds, tok.pad_token_id)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    hyp = [x.strip() for x in tok.batch_decode(preds, skip_special_tokens=True)]
    ref = [x.strip() for x in tok.batch_decode(labels, skip_special_tokens=True)]
    return {"chrf": chrf.corpus_score(hyp, [ref]).score}

args = Seq2SeqTrainingArguments(
    output_dir=str(SORTIE / "checkpoints"),
    num_train_epochs=1 if SMOKE else EPOCHS,
    max_steps=30 if SMOKE else -1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=10 if SMOKE else 300,
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    eval_strategy="steps" if SMOKE else "epoch",
    eval_steps=30 if SMOKE else None,
    save_strategy="steps" if SMOKE else "epoch",
    save_steps=30 if SMOKE else None,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,
    fp16=False,                 # m2m100 produit des nan en fp16 : on garde le float32
    logging_steps=10 if SMOKE else 50,
    report_to=[],
    seed=13,
    data_seed=13,
)
trainer = Seq2SeqTrainer(
    model=modele,
    args=args,
    train_dataset=train,
    eval_dataset=dev,
    data_collator=DataCollatorForSeq2Seq(tok, model=modele),
    processing_class=tok,
    compute_metrics=mesures,
)
point = get_last_checkpoint(args.output_dir) if Path(args.output_dir).is_dir() else None
print("reprise depuis", point)
debut = time.time()
resultat = trainer.train(resume_from_checkpoint=point)
secondes = time.time() - debut
pas = max(1, trainer.state.global_step)
print("entra?nement termin? en", round(secondes / 60, 1), "min |", round(secondes / pas, 2), "s/pas")
print("m?triques", resultat.metrics)

eval_dev = trainer.evaluate()
print("chrF dev", eval_dev)
eval_soreva = trainer.evaluate(test, metric_key_prefix="soreva")
print("chrF SOREVA (jamais vu)", eval_soreva)

FINAL = SORTIE / "final"
trainer.save_model(str(FINAL))
tok.save_pretrained(str(FINAL))
print("mod?le final :", FINAL)

## 5. Contr?les apr?s entra?nement

Les 5 phrases rapides servent aussi en mode `SMOKE` ; les 10 phrases bancaires v?rifient les termes qui comptent pour KAMA. On regarde surtout les contresens sur envoyer, confirmer, annuler et alerte.

In [ ]:
CONTROLES = [
    "Transfer completed.",
    "Please confirm the transfer.",
    "Hello, ZZ1ZZ.",
    "The beneficiary will receive the money.",
    "Your balance is insufficient.",
]
BANQUE = [
    "Enter the beneficiary account number.",
    "The transfer fee is 500 FCFA.",
    "Confirm the transfer to continue.",
    "Your transfer has been sent.",
    "The recipient will receive the money immediately.",
    "Insufficient balance.",
    "Cancel the transfer.",
    "The transaction failed. Please try again.",
    "Check the amount before confirming.",
    "Your account balance is 25,000 FCFA.",
]

def traduire(textes):
    tok.src_lang = "en"
    modele.eval()
    enc = tok(textes, return_tensors="pt", padding=True, truncation=True, max_length=128).to(modele.device)
    with torch.inference_mode():
        sortie = modele.generate(
            **enc, forced_bos_token_id=FORCED_BOS, num_beams=4, no_repeat_ngram_size=3, max_new_tokens=128
        )
    return tok.batch_decode(sortie, skip_special_tokens=True)

lignes = ["=== 5 contr?les rapides ==="]
for source, cible in zip(CONTROLES, traduire(CONTROLES)):
    lignes.append(f"EN: {source}\nPIDGIN: {cible}\n")
lignes.append("=== 10 phrases bancaires ===")
for source, cible in zip(BANQUE, traduire(BANQUE)):
    lignes.append(f"EN: {source}\nPIDGIN: {cible}\n")
texte = "\n".join(lignes)
(SORTIE / "controles.txt").write_text(texte, encoding="utf-8")
print(texte)

resultats = {
    "modele": MODELE,
    "smoke": SMOKE,
    "epochs": 1 if SMOKE else EPOCHS,
    "temps_secondes": round(secondes, 1),
    "temps_par_pas": round(secondes / pas, 3),
    "pas": pas,
    "metriques_entrainement": resultat.metrics,
    "dev": eval_dev,
    "soreva": eval_soreva,
    "controles": dict(zip(CONTROLES, traduire(CONTROLES))),
    "banque": dict(zip(BANQUE, traduire(BANQUE))),
}
(SORTIE / "resultats.json").write_text(json.dumps(resultats, ensure_ascii=False, indent=1), encoding="utf-8")

## 6. Conversion CTranslate2 int8

Le serveur ne charge que le dossier converti, pas le mod?le Hugging Face temporaire de Colab. On copie aussi le tokenizer dans le dossier CT2 : sans lui, la cha?ne locale ne peut pas relire le mod?le.

In [ ]:
import subprocess
from transformers import M2M100Tokenizer

CT2 = SORTIE / "ct2_int8"
subprocess.run([
    "ct2-transformers-converter",
    "--model", str(FINAL),
    "--output_dir", str(CT2),
    "--quantization", "int8",
    "--force",
], check=True)
M2M100Tokenizer.from_pretrained(str(FINAL)).save_pretrained(str(CT2))

import ctranslate2
tok_ct2 = M2M100Tokenizer.from_pretrained(str(CT2), local_files_only=True)
tok_ct2.src_lang = "en"
tr = ctranslate2.Translator(str(CT2), device="cpu", compute_type="int8", inter_threads=1, intra_threads=2)
entree = tok_ct2.convert_ids_to_tokens(tok_ct2.encode("Transfer completed."))
sortie = tr.translate_batch([entree], target_prefix=[["__fr__"]], beam_size=4, max_decoding_length=128)[0].hypotheses[0]
if sortie and sortie[0] == "__fr__":
    sortie = sortie[1:]
print("CT2 int8 :", tok_ct2.decode(tok_ct2.convert_tokens_to_ids(sortie), skip_special_tokens=True))
print("dossier CT2 :", CT2)

## 7. R?cup?rer les mod?les

Produit deux zips : le mod?le Hugging Face (`final`) et le mod?le CTranslate2 int8. Sur ton poste, le CT2 va dans `KAMA/models/poids/ct2/` sous un **nouveau nom** (ne remplace pas la v2 avant comparaison).

In [ ]:
import shutil, zipfile
from pathlib import Path

def zipper(source, archive, racine):
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(source).rglob("*"):
            if p.is_file():
                z.write(p, str(Path(racine) / p.relative_to(source)))

zip_hf = SORTIE / "KAMA_m2m100_hf.zip"
zip_ct2 = SORTIE / "KAMA_m2m100_ct2.zip"
zipper(FINAL, zip_hf, "final")
zipper(CT2, zip_ct2, "ct2_int8")
print(zip_hf, round(zip_hf.stat().st_size / 1048576, 1), "Mo")
print(zip_ct2, round(zip_ct2.stat().st_size / 1048576, 1), "Mo")

if os.path.isdir(MON_DRIVE):
    os.makedirs(SORTIE_DRIVE, exist_ok=True)
    for p in (zip_hf, zip_ct2, SORTIE / "resultats.json", SORTIE / "controles.txt"):
        shutil.copy(p, SORTIE_DRIVE)
    print(os.listdir(SORTIE_DRIVE))
else:
    from google.colab import files
    files.download(str(zip_hf))
    files.download(str(zip_ct2))
    files.download(str(SORTIE / "resultats.json"))
    files.download(str(SORTIE / "controles.txt"))
    print("t?l?chargements lanc?s : range-les dans Mon Drive/KAMA/pidgin_m2m_sorties")

## 8. Accepter ou refuser

On accepte seulement si :
- 100 % des valeurs (montants, noms, num?ros, balises `ZZ...ZZ`) sont conserv?es ;
- le mod?le d?passe la v2 au test in?dit et sur les termes critiques ;
- un locuteur ne trouve aucun contresens sur **envoyer**, **confirmer**, **annuler** ou une **alerte** selon le locuteur.

Comparaisons ? lancer sur le poste, depuis `KAMA/training/pidgin` :

```bash
python 03_tester_inedit.py --modele <dossier_ct2_m2m> --sortie data/rapport_inedit_m2m.csv
python 10_termes_critiques.py data/rapport_inedit_v2_ct2.csv data/rapport_inedit_m2m.csv
python 08_comparer_soreva.py v2=<dossier_ct2_v2> m2m=<dossier_ct2_m2m>
```

Le c?blage serveur (`config/routes*.json`) reste ? faire apr?s comparaison : ne pas modifier la v2 avant d'avoir les r?sultats.